# GPU Memory Hierarchy & Data Movement with CuTe DSL
**Authors: Claude Code & Pramodith**

A modern NVIDIA GPU has **5 levels of memory**, each trading off size for speed:

| Memory Type | Scope | Capacity | Latency (approx) |
|---|---|---|---|
| Registers (RMEM) | Per-thread | 255 registers/thread | ~1 cycle |
| L1 Cache | Per-SM | Up to 256 KB (shared with SMEM) | ~33 cycles |
| Shared Memory (SMEM) | Per-block | Up to 228 KB | ~30 cycles |
| L2 Cache | Device-wide | 50 MB (partitioned) | ~260 cycles (near-hit), up to ~420 cycles (far-hit) |
| Global Memory (GMEM) | Device-wide | 80 GB HBM3 | ~550-800 cycles |

*Figures based on the H100 SXM5 GPU (Hopper architecture). L1 and SMEM share the same 256 KB physical SRAM per SM with a configurable split. L2 latency varies due to Hopper's two-partition design. accesses to the local partition are faster than cross-partition accesses. Sources: Luo et al. "Dissecting the NVIDIA Hopper Architecture through Microbenchmarking" (2025), Chips and Cheese "Nvidia's H100: Funny L2, and Tons of Bandwidth" (2023), NVIDIA H100 Architecture Whitepaper.*


**Three of these are programmer-controlled:** Global Memory (GMEM), Shared Memory (SMEM), and Register Memory (RMEM). The L1 and L2 caches are hardware-managed data flows through them automatically, though we can influence their behavior with cache hints.

The fundamental job of a high-performance GPU kernel is to **move data up this hierarchy** as efficiently as possible: from GMEM into SMEM (shared across a thread block), then from SMEM into RMEM (private to each thread), where the actual computation happens.

CuTe DSL provides **CopyAtoms**, abstractions over the hardware copy instructions to express these data movement patterns. This notebook explores each memory level and demonstrates how to move data between them.

In [ ]:
%pip install -q torch triton nvidia-cutlass nvidia-cutlass-dsl

In [ ]:
import os
import torch
import triton

# Must be set BEFORE importing cutlass, the EnvManager reads env vars at import time
os.environ["CUTE_DSL_KEEP_PTX"] = "1"

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

assert torch.cuda.is_available(), "CUDA GPU required"
major, minor = torch.cuda.get_device_capability()
os.environ["CUTE_DSL_ARCH"] = f"sm_{major}{minor}" + ("a" if major >= 9 else "")

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute capability: sm_{major}{minor}")
print(f"Target arch: {os.environ['CUTE_DSL_ARCH']}")

GPU: NVIDIA GeForce RTX 4070 SUPER
Compute capability: sm_89
Target arch: sm_89


## The 5 Memory Levels in Detail

### 1. Global Memory (GMEM)
- **Scope:** Visible to all threads on the device
- **Backed by:** HBM (High Bandwidth Memory) or GDDR
- **Size:** GBs (e.g., 12 GB on RTX 4070, 80 GB on A100/H100)
- **Access pattern:** Coalesced 128-byte transactions. When 32 threads in a warp access consecutive addresses, the hardware merges them into minimal transactions. Scattered access wastes bandwidth.

This is where your PyTorch tensors live. Every kernel starts and ends by reading from and writing to GMEM.

### 2. L2 Cache
- **Scope:** Device-wide (shared by all SMs)
- **Size:** MBs (e.g., 36 MB on RTX 4070, 40 MB on A100, 50 MB on H100)
- **Managed by:** Hardware (transparent caching of GMEM accesses)
- **Programmer influence:** Cache eviction hints (`EVICT_FIRST`, `EVICT_LAST`, `EVICT_NORMAL`) via CopyAtom parameters, and L2 persistence controls via `cudaAccessPolicyWindow`

### 3. Shared Memory (SMEM)
- **Scope:** Per-thread-block (all threads in a block see the same SMEM)
- **Size:** Configurable per-block, up to 228 KB on Ampere/Hopper
- **Key feature:** Software-managed scratchpad. The programmer explicitly allocates and fills it.
- **Why it matters:** When multiple threads in a block need the same data, loading it once into SMEM and reading it N times is far cheaper than N separate GMEM loads.

### 4. L1 Cache
- **Scope:** Per-SM
- **Size:** 128-256 KB (shared/configurable with SMEM on some architectures)
- **Managed by:** Hardware (caches GMEM and register spills)
- **Note:** On modern GPUs (Ampere+), L1 and SMEM share the same on-chip SRAM. The split is configurable via `cudaFuncSetAttribute`.

### 5. Register Memory (RMEM)
- **Scope:** Per-thread (completely private)
- **Size:** Up to 255 32-bit registers per thread
- **Key feature:** This is where computation actually happens. Instructions like FMA read their operands from registers and write results back to registers.
- **Tradeoff:** More registers per thread = fewer threads per SM (lower occupancy). The compiler manages register allocation, but CuTe DSL's `make_rmem_tensor` lets you explicitly allocate register-backed tensors.

## Data Movement Paths

Not all memory-to-memory paths are equal. The GPU hardware provides specialized instructions for certain paths:

```
  GMEM ──────────────────────────────────> RMEM     (LD.GLOBAL, load through L1/L2)
  GMEM ──────────────────> SMEM                     (CP.ASYNC, bypasses registers!)
                           SMEM ────────> RMEM      (LDS, load from shared memory)
                           SMEM ────────> RMEM      (LDMATRIX, warp-level structured load)
  GMEM <──────────────────────────────── RMEM       (ST.GLOBAL, store through L1/L2)
                           SMEM <──────── RMEM      (STS, store to shared memory)
```

The key insight is that **GMEM → SMEM can bypass registers entirely** using `cp.async` (Ampere+). This is important because registers are a scarce resource. Using them as a waypoint for data that's just passing through to SMEM is wasteful.

In CuTe DSL, each of these hardware paths is represented by a **CopyAtom**, a type that encapsulates the instruction, its operand layout, and how threads cooperate to move data. Let's explore each path.

## Path 1: GMEM → RMEM → GMEM (Direct Load/Store)

The simplest data movement pattern: each thread loads data from global memory directly into its registers, computes on it, and stores results back. This is what happens when you index into a GMEM tensor inside a kernel.

Under the hood, the GPU issues `LD.GLOBAL` instructions that travel through L2 → L1 → registers. We don't need to explicitly manage SMEM at all.

Let's write a simple kernel that loads elements from GMEM into register-backed tensors using `make_rmem_tensor`, doubles them, and writes them back.

In [ ]:
ELEMS_PER_THREAD = 4

@cute.kernel
def gmem_to_rmem_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Each thread loads ELEMS_PER_THREAD elements from GMEM into registers,
    doubles them, and stores back to GMEM."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()

    global_tid = bidx * bdim + tidx

    # Allocate a register-backed tensor (RMEM) to hold this thread's data
    rmem = cute.make_rmem_tensor(ELEMS_PER_THREAD, cutlass.Float32)

    # GMEM → RMEM: load elements into registers
    for i in range(ELEMS_PER_THREAD):
        rmem[i] = gIn[global_tid * ELEMS_PER_THREAD + i]

    # Compute in registers (doubling each element)
    for i in range(ELEMS_PER_THREAD):
        rmem[i] = rmem[i] * 2.0

    # RMEM → GMEM: store results back
    for i in range(ELEMS_PER_THREAD):
        gOut[global_tid * ELEMS_PER_THREAD + i] = rmem[i]


@cute.jit
def gmem_to_rmem(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    threads_per_block = 256
    blocks = N // (threads_per_block * ELEMS_PER_THREAD)
    gmem_to_rmem_kernel(mIn, mOut).launch(
        grid=(blocks, 1, 1),
        block=(threads_per_block, 1, 1),
    )


# Test correctness
N = 1 << 20  # ~1M elements
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

gmem_to_rmem_fn = cute.compile(gmem_to_rmem, inp_, out_)
gmem_to_rmem_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")
print(f"  PTX:\n{gmem_to_rmem_fn.__ptx__}")

GMEM → RMEM → GMEM: PASSED (N=1,048,576)
  Input sample:  [-2.3257641792297363, -0.8461601734161377, 1.3627749681472778, 0.5746921896934509]
  Output sample: [-4.651528358459473, -1.6923203468322754, 2.7255499362945557, 1.1493843793869019]
  PTX:
//
// Generated by NVIDIA NVVM Compiler
//
// Compiler Build ID: CL-36006120
// Cuda compilation tools, release 12.9, V12.9.83
// Based on NVVM 20.0.0
//

.version 8.8
.target sm_89
.address_size 64

	// .globl	kernel_cutlass_gmem_to_rmem_kernel_tensorptrf32gmemalign16o10485761_tensorptrf32gmemalign16o10485761_0

.visible .entry kernel_cutlass_gmem_to_rmem_kernel_tensorptrf32gmemalign16o10485761_tensorptrf32gmemalign16o10485761_0(
	.param .align 8 .b8 kernel_cutlass_gmem_to_rmem_kernel_tensorptrf32gmemalign16o10485761_tensorptrf32gmemalign16o10485761_0_param_0[8],
	.param .align 8 .b8 kernel_cutlass_gmem_to_rmem_kernel_tensorptrf32gmemalign16o10485761_tensorptrf32gmemalign16o10485761_0_param_1[8]
)
.reqntid 256, 1, 1
{
	.reg .b32 	%r<6>;
	.reg

### What happened under the hood

Let's walk through the PTX the compiler generated and map each section back to the Python kernel.

**Register declarations**
```
.reqntid 256, 1, 1          ← block=(256, 1, 1) from our launch config
.reg .b32  %r<6>;           ← 6 × 32-bit integer registers (%r0–%r5)
.reg .f32  %f<9>;           ← 9 × 32-bit float registers (%f0–%f8)
.reg .b64  %rd<6>;          ← 6 × 64-bit registers (%rd0–%rd5, for pointers / addresses)
```
The `.reg .f32 %f<9>` syntax declares registers `%f0` through `%f8` the `<N>` range always starts at 0, so `%f0` is declared but never used. This has no runtime cost; the hardware register allocator simply ignores it. Of the 8 that matter: `%f1–%f4` hold the loaded values and `%f5–%f8` hold the doubled results, because both sets are live at the same time. No shared memory (`.shared`) is declared anywhere, this is purely GMEM → registers → GMEM.

---

**Computing the global thread index**
```
mov.u32      %r1, %tid.x;            ← r1 = threadIdx.x
mov.u32      %r2, %ctaid.x;          ← r2 = blockIdx.x
mov.u32      %r3, %ntid.x;           ← r3 = blockDim.x  (= 256)
mad.lo.s32   %r4, %r2, %r3, %r1;    ← r4 = blockIdx.x * 256 + threadIdx.x = global_tid
```
This is `global_tid = bidx * bdim + tidx` from the kernel, compiled to a single multiply-add (`mad`). `.lo` means "keep the low 32 bits of the product" which is all we need since the result fits in 32 bits.

---

**GMEM → RMEM: the four loads** (maps to `rmem[i] = gIn[global_tid * ELEMS_PER_THREAD + i]`)
```
add.s64       %rd4, %rd2, %rd3;      ← rd4 = gIn + byte_offset = &gIn[global_tid × 4]
ld.global.f32 %f1, [%rd4];           ← f1 = gIn[base + 0]
ld.global.f32 %f2, [%rd4+4];         ← f2 = gIn[base + 1]   (+4 bytes)
ld.global.f32 %f3, [%rd4+8];         ← f3 = gIn[base + 2]   (+8 bytes)
ld.global.f32 %f4, [%rd4+12];        ← f4 = gIn[base + 3]   (+12 bytes)
```
The `for i in range(4)` loop is **fully unrolled**. There's no loop counter or branch, just four consecutive load instructions with constant offsets (`+0`, `+4`, `+8`, `+12` bytes). Each `ld.global.f32` issues a request that travels: **GMEM → L2 → L1 → register file**.

---

**Compute in registers** (maps to `rmem[i] = rmem[i] * 2.0`)
```
add.f32  %f5, %f1, %f1;              ← f5 = f1 + f1 = f1 × 2.0
add.f32  %f6, %f2, %f2;              ← f6 = f2 + f2 = f2 × 2.0
add.f32  %f7, %f3, %f3;              ← f7 = f3 + f3 = f3 × 2.0
add.f32  %f8, %f4, %f4;              ← f8 = f4 + f4 = f4 × 2.0
```
The compiler optimized `× 2.0` into `add.f32 %f, %f, %f` (x + x) instead of emitting a multiply. Addition and multiplication have the same throughput on modern NVIDIA GPUs, but this avoids loading the constant `2.0` into a register. These are **pure register operations**: both operands and the result live in the register file with zero memory traffic.

---

**RMEM → GMEM: the four stores** (maps to `gOut[global_tid * ELEMS_PER_THREAD + i] = rmem[i]`)
```
add.s64       %rd5, %rd1, %rd3;      ← rd5 = gOut + byte_offset = &gOut[global_tid × 4]
st.global.f32 [%rd5], %f5;           ← gOut[base + 0] = f5
st.global.f32 [%rd5+4], %f6;         ← gOut[base + 1] = f6
st.global.f32 [%rd5+8], %f7;         ← gOut[base + 2] = f7
st.global.f32 [%rd5+12], %f8;        ← gOut[base + 3] = f8
```
Same unrolled pattern as the loads. Each `st.global.f32` sends data: **register file → L1 → L2 → GMEM**.

---

_Note: We could've also just done `gOut[...] = gIn[...] * 2.0` without using `make_rmem_tensor` at all, and the compiler would still generate the same code. The explicit RMEM tensor just makes it clearer that these values are register-resident and not spilled to memory._

## Path 2: GMEM → SMEM → RMEM (The Two-Stage Pattern)

When threads within a block need overlapping data, we can save GMEM bandwidth by:
1. **Loading data from GMEM into SMEM** once (cooperatively across all threads in the block)
2. **Having each thread read from SMEM** into its registers

This is the bread-and-butter pattern for GEMM, convolution, reduction, and stencil kernels.

In the kernel below, we demonstrate this explicitly:
- Each thread cooperatively loads part of a block-sized chunk from GMEM into SMEM
- After a `__syncthreads()`, each thread reads from SMEM into registers
- The thread computes on registers and writes results back to GMEM

In [ ]:
BLOCK_SIZE = 256
ELEMS_PER_THREAD_SMEM = 4
SMEM_SIZE_NAIVE = BLOCK_SIZE * ELEMS_PER_THREAD_SMEM

@cute.kernel
def gmem_smem_rmem_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Demonstrates the GMEM → SMEM → RMEM → GMEM data movement pattern.
    Each thread block cooperatively loads a chunk into SMEM, then each thread
    reads its elements from SMEM into registers, computes, and writes back."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # Step 1: Allocate shared memory for the block (SMEM)
    smem_ptr = cute.arch.alloc_smem(cutlass.Float32, SMEM_SIZE_NAIVE, alignment=16)
    smem = cute.make_tensor(smem_ptr, SMEM_SIZE_NAIVE)

    # Step 2: GMEM → SMEM: each thread loads ELEMS_PER_THREAD_SMEM elements cooperatively
    base_global = bidx * SMEM_SIZE_NAIVE + tidx * ELEMS_PER_THREAD_SMEM
    base_smem = tidx * ELEMS_PER_THREAD_SMEM
    for i in range(ELEMS_PER_THREAD_SMEM):
        smem[base_smem + i] = gIn[base_global + i]

    # Step 3: Synchronize, ensure all threads have finished writing to SMEM
    cute.arch.sync_threads()

    # Step 4: SMEM → RMEM: each thread reads its elements into registers
    rmem = cute.make_rmem_tensor(ELEMS_PER_THREAD_SMEM, cutlass.Float32)
    for i in range(ELEMS_PER_THREAD_SMEM):
        rmem[i] = smem[base_smem + i]

    # Step 5: Compute in registers
    for i in range(ELEMS_PER_THREAD_SMEM):
        rmem[i] = rmem[i] * 2.0

    # Step 6: RMEM → GMEM: write results back
    for i in range(ELEMS_PER_THREAD_SMEM):
        gOut[base_global + i] = rmem[i]


@cute.jit
def gmem_smem_rmem(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    gmem_smem_rmem_kernel(mIn, mOut).launch(
        grid=(N // SMEM_SIZE_NAIVE, 1, 1),
        block=(BLOCK_SIZE, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

gmem_smem_rmem_fn = cute.compile(gmem_smem_rmem, inp_, out_)
gmem_smem_rmem_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → SMEM → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")

GMEM → SMEM → RMEM → GMEM: PASSED (N=1,048,576)
  Input sample:  [-0.6692324876785278, -0.7879005670547485, -0.08097366243600845, 1.2964977025985718]
  Output sample: [-1.3384649753570557, -1.575801134109497, -0.1619473248720169, 2.5929954051971436]


### Breaking down the SMEM pattern

The key CuTe DSL primitives for shared memory:

| Step | CuTe DSL Call | What it does |
|------|--------------|--------------|
| Allocate SMEM | `cute.arch.alloc_smem(dtype, num_elems, alignment)` | Statically allocates a block of shared memory, returns a pointer |
| Create SMEM tensor | `cute.make_tensor(smem_ptr, shape)` | Wraps the SMEM pointer with a layout to get an indexable tensor |
| GMEM → SMEM | `smem[tidx] = gIn[global_idx]` | Each thread loads one element (travels GMEM → L2 → L1 → RMEM → SMEM) |
| Synchronize | `cute.arch.sync_threads()` | Barrier that ensures all threads have finished their SMEM writes |
| SMEM → RMEM | `rmem[0] = smem[tidx]` | Thread reads from SMEM into a register |

Note that the naive `smem[tidx] = gIn[idx]` path actually goes **GMEM → registers → SMEM** (two hops). The data **briefly passes through registers** because the basic load/store instructions require register operands. On Ampere+, `cp.async` can bypass this register waypoint, we'll see that next.

### A note on alignment

Both `from_dlpack(tensor, assumed_align=N)` and `alloc_smem(dtype, n, alignment=N)` take an alignment parameter. **Alignment means the base memory address is guaranteed to be a multiple of N bytes.** This matters because the compiler can emit wider, faster instructions when it knows the alignment:

| Alignment | Widest Instruction | Bytes/op | FP32 elements/op |
|-----------|-------------------|----------|-------------------|
| 4 bytes   | `LD.GLOBAL.32`    | 4        | 1                 |
| 8 bytes   | `LD.GLOBAL.64`    | 8        | 2                 |
| 16 bytes  | `LD.GLOBAL.128`   | 16       | 4                 |

**16 bytes is the sweet spot** since `LD.GLOBAL.128` is the widest standard load the GPU has, so alignment beyond 16 doesn't help for regular GMEM ↔ RMEM paths.

## Path 3: GMEM → SMEM via `cp.async` (Register-Free Transfer)

Starting with Ampere (SM80), NVIDIA introduced `cp.async`. An instruction that copies data directly from global memory to shared memory **without using registers as intermediaries**. This has two benefits:

1. **Saves registers:** The data never touches the register file, leaving more registers for computation.
2. **Asynchronous:** The copy is initiated and the thread can continue executing other instructions. The thread only waits when it actually needs the data in SMEM.

In CuTe DSL, this path uses `CopyG2SOp` (Copy Global-to-Shared Operation):

```python
op = cute.nvgpu.cpasync.CopyG2SOp()
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)
```

The `num_bits_per_copy` parameter controls the transaction size. 128 bits means 4 FP32 elements per copy operation per thread. Because these copies are asynchronous, we need a way to know when they've landed in SMEM, that's where **commit groups** come in. We call `cute.arch.cp_async_commit_group()` to bundle pending copies into a trackable group, then `cute.arch.cp_async_wait_group(0)` to wait for all groups to complete.

In [ ]:
BLOCK_SIZE_ASYNC = 256
ELEMS_PER_THREAD_ASYNC = 4
SMEM_SIZE = BLOCK_SIZE_ASYNC * ELEMS_PER_THREAD_ASYNC  # 1024 elements per block

@cute.kernel
def cp_async_kernel(
    gIn: cute.Tensor,
    gOut: cute.Tensor,
):
    """Demonstrates GMEM → SMEM via cp.async, then SMEM → RMEM → GMEM."""
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # Allocate SMEM
    smem_ptr = cute.arch.alloc_smem(cutlass.Float32, SMEM_SIZE, alignment=16)
    smem = cute.make_tensor(smem_ptr, SMEM_SIZE)

    # Build the cp.async copy atom (32-bit per operation)
    # Note: 128-bit cp.async requires statically provable 16-byte alignment,
    # which needs proper tiling infrastructure (covered in a future notebook).
    # Here we use 32-bit granularity to demonstrate the async mechanism.
    cp_async_op = cute.nvgpu.cpasync.CopyG2SOp()
    cp_async_atom = cute.make_copy_atom(cp_async_op, cutlass.Float32, num_bits_per_copy=32)

    # Each thread is responsible for ELEMS_PER_THREAD_ASYNC contiguous elements
    base_idx = bidx * SMEM_SIZE + tidx * ELEMS_PER_THREAD_ASYNC
    smem_offset = tidx * ELEMS_PER_THREAD_ASYNC

    # GMEM → SMEM via cp.async (bypasses registers!)
    # Issue one 32-bit async copy per element
    for i in range(ELEMS_PER_THREAD_ASYNC):
        gmem_elem = cute.make_tensor(
            gIn.iterator + base_idx + i,
            cute.make_layout((1, 1), stride=(0, 1))
        )
        smem_elem = cute.make_tensor(
            smem_ptr + smem_offset + i,
            cute.make_layout((1, 1), stride=(0, 1))
        )
        cute.copy(cp_async_atom, gmem_elem, smem_elem)

    # Commit and wait for async copy to complete
    cute.arch.cp_async_commit_group()
    cute.arch.cp_async_wait_group(0)
    cute.arch.sync_threads()

    # SMEM → RMEM: load into registers for computation
    rmem = cute.make_rmem_tensor(ELEMS_PER_THREAD_ASYNC, cutlass.Float32)
    for i in range(ELEMS_PER_THREAD_ASYNC):
        rmem[i] = smem[smem_offset + i]

    # Compute in registers
    for i in range(ELEMS_PER_THREAD_ASYNC):
        rmem[i] = rmem[i] * 2.0

    # RMEM → GMEM
    for i in range(ELEMS_PER_THREAD_ASYNC):
        gOut[base_idx + i] = rmem[i]


@cute.jit
def cp_async_demo(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    cp_async_kernel(mIn, mOut).launch(
        grid=(N // SMEM_SIZE, 1, 1),
        block=(BLOCK_SIZE_ASYNC, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

cp_async_fn = cute.compile(cp_async_demo, inp_, out_)
cp_async_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"GMEM → SMEM (cp.async) → RMEM → GMEM: PASSED (N={N:,})")
print(f"  Input sample:  {inp[:4].tolist()}")
print(f"  Output sample: {out[:4].tolist()}")

GMEM → SMEM (cp.async) → RMEM → GMEM: PASSED (N=1,048,576)
  Input sample:  [1.4212433099746704, 0.4171926975250244, 0.04456259682774544, -2.218416929244995]
  Output sample: [2.842486619949341, 0.8343853950500488, 0.08912519365549088, -4.43683385848999]


### cp.async vs naive GMEM → SMEM

The difference between the naive path and `cp.async`:

```
Naive:     GMEM  →  Registers  →  SMEM     (2 hops, uses registers as temporary)
cp.async:  GMEM  →  SMEM                   (1 hop, direct DMA by the memory subsystem)
```

**`cp.async` advantages:**
- **Frees up registers**: data doesn't occupy register slots during transit
- **Asynchronous**: the thread issues the copy and continues doing other work
- **Higher throughput**: the memory controller can pipeline multiple 128-bit async copies

**Commit groups and the async handshake:**

A **commit group** is a batch of async copy operations that we want to track as a single unit. Every `cp.async` copy issued by a thread is implicitly appended to that thread's current (open) group. Calling `cp_async_commit_group()` seals the current group. No more copies can join it, and opens a fresh one for subsequent copies. You can then wait for groups to complete by count:

```
cp.async  ...                     ┐
cp.async  ...                     ├── commit group 0
cp.async  ...                     ┘
cp_async_commit_group()           ← seal group 0

cp.async  ...                     ┐
cp.async  ...                     ├── commit group 1
cp_async_commit_group()           ← seal group 1

cp_async_wait_group(N)            ← wait until at most N groups are still in-flight
```

| Call | Meaning |
|------|--------|
| `cp_async_wait_group(0)` | Wait for **all** groups to complete |
| `cp_async_wait_group(1)` | Wait until at most 1 group is still in-flight (all older groups are done) |

In the kernel above we only have one batch of copies, so a single commit + `wait_group(0)` is all we need. But in software-pipelined kernels (like tiled GEMM), you'd issue multiple groups and use `wait_group(1)` to overlap loading the next tile with computing on the current one.

**The handshake in CuTe DSL:**
1. `cute.copy(cp_async_atom, src, dst)`: initiates the async copy (appended to current group)
2. `cute.arch.cp_async_commit_group()`: seals the current group
3. `cute.arch.cp_async_wait_group(N)`: waits until at most N groups are still in-flight
4. `cute.arch.sync_threads()`: ensures all threads in the block have completed the wait before reading SMEM

Let's prove that the `async_copy` command uses `ELEMS_PER_THREAD` fewer float registers than the naive SMEM load by parsing the ptx of both.

In [ ]:
import re

def count_float_registers(ptx: str) -> int:
    """Count float register declarations (.f32, .f64) in PTX."""
    total = 0
    for match in re.finditer(r'\.reg\s+\.(f\d+)\s+%\w+<(\d+)>', ptx):
        total += int(match.group(2))
    return total

naive_fregs = count_float_registers(gmem_smem_rmem_fn.__ptx__)
async_fregs = count_float_registers(cp_async_fn.__ptx__)

diff = async_fregs - naive_fregs
sign = '+' if diff > 0 else ''
print('Float register comparison (both kernels: 4 elements/thread):')
print(f'  Path 2 (naive):     {naive_fregs} float registers')
print(f'  Path 3 (cp.async):  {async_fregs} float registers  ({sign}{diff})')
print()
print(f'cp.async saves {abs(diff)} float registers by bypassing the register file during GMEM → SMEM transfer.')

Float register comparison (both kernels: 4 elements/thread):
  Path 2 (naive):     13 float registers
  Path 3 (cp.async):  9 float registers  (-4)

cp.async saves 4 float registers by bypassing the register file during GMEM → SMEM transfer.


## Using `autovec_copy` for Automatic Vectorization

CuTe DSL provides `autovec_copy`, a copy function that analyzes the pointer alignment and layout of source and destination tensors to automatically select the widest safe vector width. Under the hood, it creates a `CopyUniversalOp` atom with the optimal `num_bits_per_copy`.

This is the simplest way to get vectorized copies between any memory spaces:

```python
cute.autovec_copy(src_tensor, dst_tensor)
```

`autovec_copy` examines:
1. The **layout alignment** (stride patterns) of both tensors
2. The **pointer alignment** (byte alignment of the base address)
3. Caps at 256 bits maximum

Let's use it to copy between GMEM and RMEM with automatic vectorization.

In [ ]:
ELEMS_PER_THREAD_VEC = 4

@cute.kernel
def autovec_kernel(gIn: cute.Tensor, gOut: cute.Tensor):
    """Uses autovec_copy for automatic vectorization of GMEM↔RMEM copies.

    Key: we use local_tile to extract each thread's slice instead of
    raw pointer arithmetic (gIn.iterator + offset). Pointer arithmetic
    loses the alignment annotation from assumed_align=16, so autovec_copy
    can't prove the address is 128-bit aligned and falls back to scalar loads.
    local_tile keeps the original base pointer and encodes the offset in the
    layout, preserving alignment info for the vectorizer.
    """
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    global_tid = bidx * bdim + tidx

    # Extract this thread's (4,) tile alignment metadata is preserved
    src  = cute.local_tile(gIn,  (ELEMS_PER_THREAD_VEC,), (global_tid,))
    rmem = cute.make_rmem_tensor((ELEMS_PER_THREAD_VEC,), cutlass.Float32)

    # GMEM → RMEM: autovec_copy picks the widest safe vector width
    cute.autovec_copy(src, rmem)

    # Compute in registers
    for i in range(ELEMS_PER_THREAD_VEC):
        rmem[i] = rmem[i] * 2.0

    # RMEM → GMEM
    dst = cute.local_tile(gOut, (ELEMS_PER_THREAD_VEC,), (global_tid,))
    cute.autovec_copy(rmem, dst)


@cute.jit
def autovec_demo(mIn: cute.Tensor, mOut: cute.Tensor):
    N = mIn.shape[0]
    threads_per_block = 256
    blocks = N // (threads_per_block * ELEMS_PER_THREAD_VEC)
    autovec_kernel(mIn, mOut).launch(
        grid=(blocks, 1, 1),
        block=(threads_per_block, 1, 1),
    )


# Test correctness
N = 1 << 20
inp = torch.randn(N, device="cuda", dtype=torch.float32)
out = torch.zeros(N, device="cuda", dtype=torch.float32)

inp_ = from_dlpack(inp, assumed_align=16)
out_ = from_dlpack(out, assumed_align=16)

autovec_fn = cute.compile(autovec_demo, inp_, out_)
autovec_fn(inp_, out_)

torch.testing.assert_close(out, inp * 2.0)
print(f"autovec_copy GMEM → RMEM → GMEM: PASSED (N={N:,})")

autovec_copy GMEM → RMEM → GMEM: PASSED (N=1,048,576)


### Why vectorization matters

Under the hood, `autovec_copy` selects the widest load/store instruction that the pointer alignment allows. Wider vector loads issue fewer instructions for the same data:

| Vector Width | PTX Instruction | Bytes/instruction | FP32 Elements | FP16/BF16 Elements |
|-------------|----------------|-------------------|---------------|---------------------|
| 32-bit | `LD.GLOBAL.B32` | 4 | 1 | 2 |
| 64-bit | `LD.GLOBAL.B64` | 8 | 2 | 4 |
| 128-bit | `LD.GLOBAL.B128` | 16 | 4 | 8 |

Fewer instructions means less scheduling overhead and better memory bus utilization. The `autovec_copy` function handles this automatically based on the tensor's alignment properties. We can confirm that we use 128 bit store/loads in the `autovec` kernel by parsing the ptx and searching for `st.global.v4.f32` and `ld.global.v4.f32` instructions as seen in the code below. 

In [ ]:
import re

ptx = autovec_fn.__ptx__

# Find all ld.global and st.global instructions, capturing the width suffix
loads  = re.findall(r'ld\.global\.(\S+)', ptx)
stores = re.findall(r'st\.global\.(\S+)', ptx)

print('Load instructions in autovec_kernel PTX:')
for instr in loads:
    print(f'  ld.global.{instr}')

print()
print('Store instructions in autovec_kernel PTX:')
for instr in stores:
    print(f'  st.global.{instr}')

print()
has_b128 = any('b128' in s or 'v4' in s for s in loads + stores)
has_b32  = any(s.startswith('b32') or s.startswith('f32') for s in loads + stores)
if has_b128 and not has_b32:
    print('autovec_copy selected 128-bit (B128) loads/stores 4× fewer instructions than scalar B32.')
elif has_b128 and has_b32:
    print('Mix of B128 and B32 instructions found.')
else:
    print('No B128 instructions found, check pointer alignment.')

Load instructions in autovec_kernel PTX:
  ld.global.v4.f32

Store instructions in autovec_kernel PTX:
  st.global.v4.f32

autovec_copy selected 128-bit (B128) loads/stores 4× fewer instructions than scalar B32.


## Why CopyAtoms? From Scalar Loads to Hardware-Mapped Instructions

The manual copy loop in Path 1 and even `autovec_copy` are both **general-purpose**. They work anywhere but give you limited control over which hardware instruction is actually emitted. CopyAtoms give you **explicit, hardware-mapped control**: you pick the instruction, the width, and the memory path. The compiler doesn't guess.

### The three levels of copy abstraction

| Approach | Control | Instruction |
|----------|---------|-------------|
| Manual loop (`rmem[i] = gIn[...]`) | None, compiler decides | Scalar `LD.GLOBAL` per element |
| `autovec_copy` | Automatic, inferred from alignment | Widest *safe* vector load |
| `CopyAtom` | Explicit, you specify the op and width | Exactly the instruction you name |

### What the manual loop actually generates

The loop in `gmem_to_rmem_kernel` produces **one load per element** i.e. four separate 32-bit loads for four floats:

```ptx
ld.global.f32  %f1, [%rd4];
ld.global.f32  %f2, [%rd4+4];
ld.global.f32  %f3, [%rd4+8];
ld.global.f32  %f4, [%rd4+12];
```

A `CopyUniversalOp` atom with `num_bits_per_copy=128` collapses these into a single **vectorized load** similar to `autovec_copy`:

```ptx
ld.global.v4.f32  {%f1, %f2, %f3, %f4}, [%rd4];
```

### Pointer arithmetic kills alignment annotations

`autovec_copy` is described as selecting *the widest safe width based on pointer alignment*. "Safe" is key: if the compiler can only prove 32-bit alignment, it falls back to scalar loads silently. An explicit CopyAtom at 128-bit **fails loudly** if alignment isn't provably 128 bits.

The catch: raw pointer arithmetic (`gIn.iterator + offset`) drops the alignment annotation from `assumed_align=16` down to the natural element alignment (32 bits for fp32). **Both `autovec_copy` and an explicit atom will therefore use scalar loads when the sub-tensor is created this way.**

The fix is `cute.local_tile`: it encodes the per-thread offset in the *layout* rather than adding to the pointer, keeping the base address and letting CuTe derive alignment from the stride:

```
stride alignment = 4 fp32 × 32 bits = 128 bits  ✓  (matches assumed_align=16)
```

### Why go beyond `autovec_copy`?

With proper alignment, `autovec_copy` and `CopyUniversalOp(128)` produce identical PTX. The value of explicit CopyAtoms is **unlocking hardware paths that `autovec_copy` cannot express**:

| CopyAtom | What it unlocks |
|----------|-----------------|
| `CopyUniversalOp` (explicit) | Fails loudly on bad alignment; composable with CuTe tiling |
| `cpasync.CopyG2SOp` | Async GMEM→SMEM that bypasses registers entirely (Path 3 above) |
| `warp.LdMatrix` | Loads 8×8 matrix tiles in the swizzled layout tensor cores expect |
| `cpasync.CopyBulkTensorTile*` | TMA: hardware-managed multi-dimensional tiled copies |

## CuTe DSL Copy Atom Reference

CuTe DSL provides a rich set of **CopyAtom** types, each mapping to a specific hardware copy instruction. Let's walk through some of the copy operations available in the `cutlass.cute` package.

### How Copy Atoms Work

A CopyAtom wraps a hardware instruction and defines:
- **The source and destination memory spaces** (GMEM, SMEM, RMEM)
- **The data layout** each thread expects (how many elements, what pattern)
- **The vector width** (how many bits per copy instruction)

Usage pattern:
```python
# 1. Create a CopyOp (describes the hardware instruction)
op = cute.nvgpu.CopyUniversalOp()

# 2. Create a CopyAtom from the op (adds type and width info)
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)

# 3. Execute the copy. tiles the atom across the full tensor automatically
cute.copy(atom, src_tensor, dst_tensor)
```

### Copy Functions

| Function | Description |
|----------|-------------|
| `cute.copy(atom, src, dst)` | Main copy function. Takes an atom and tiles it across the full src/dst tensors, handling multi-mode layouts automatically. Supports predication via `pred=` kwarg. |
| `cute.basic_copy(src, dst)` | Simple element-wise copy. No atom needed, uses SIMT sync copy internally. |
| `cute.basic_copy_if(pred, src, dst)` | Predicated element-wise copy. Copies `src[i]` to `dst[i]` only where `pred[i]` is true. |
| `cute.autovec_copy(src, dst)` | Auto-vectorizing copy. Analyzes layout alignment to pick the widest safe vector width automatically. |
| `cute.prefetch(atom, src)` | Prefetches data from GMEM into L2 cache. Currently only supports TMA prefetch atoms. |

### 1. Universal Copy, `cute.nvgpu.CopyUniversalOp`

**Path:** Any → Any (GMEM↔RMEM, SMEM↔RMEM, GMEM↔SMEM via registers)
**Architecture:** All (SM50+)
**PTX:** `LD.GLOBAL`, `LD.SHARED`, `ST.GLOBAL`, `ST.SHARED`, etc.

The general-purpose copy. Maps to standard load/store instructions. Supports vectorization up to 256 bits.

```python
op = cute.nvgpu.CopyUniversalOp()
atom = cute.make_copy_atom(
    op,
    cutlass.Float32,             # element type (determines layout)
    num_bits_per_copy=128,       # vectorization: 32, 64, 128, or 256 bits (0 = auto)
    l1c_evict_priority=cute.nvgpu.CacheEvictionPriority.EVICT_NORMAL,  # L1 cache hint
    memory_order=cute.nvgpu.MemoryOrder.WEAK,          # memory ordering
    memory_scope=cute.nvgpu.MemoryScope.CTA,           # visibility scope
    invariant=False,             # True = use read-only cache (LDG)
)
```

**Parameters:**
| Parameter | Options | Description |
|-----------|---------|-------------|
| `num_bits_per_copy` | 0, 32, 64, 128, 256 | Bits per copy instruction. 0 = auto-vectorize |
| `l1c_evict_priority` | `EVICT_NORMAL`, `EVICT_FIRST`, `EVICT_LAST`, `EVICT_UNCHANGED`, `NO_ALLOCATE` | L1 cache eviction hint |
| `memory_order` | `WEAK`, `RELAXED`, `ACQUIRE`, `RELEASE`, `ACQ_REL`, `SC`, `MMIO`, `CONSTANT`, `VOLATILE` | Memory ordering semantics |
| `memory_scope` | `CTA`, `CLUSTER`, `GPU`, `SYS` | Visibility scope for ordering |
| `invariant` | `True`/`False` | Read-only optimization (texture cache path) |

### 2. Asynchronous GMEM → SMEM, `cute.nvgpu.cpasync.CopyG2SOp`

**Path:** GMEM → SMEM (bypasses registers)
**Architecture:** SM80+ (Ampere)
**PTX:** `cp.async`

Initiates an asynchronous copy from global memory directly to shared memory. The thread does not wait for completion, you must explicitly commit and wait.

```python
op = cute.nvgpu.cpasync.CopyG2SOp(
    cache_mode=cute.nvgpu.cpasync.LoadCacheMode.ALWAYS  # L1 caching policy
)
atom = cute.make_copy_atom(op, cutlass.Float32, num_bits_per_copy=128)
```

**Cache modes:**
| Mode | Description |
|------|-------------|
| `ALWAYS` | Cache in L1 (default) |
| `GLOBAL` | Cache in L2 only, bypass L1 |
| `STREAMING` | Streaming access, evict first |
| `LAST_USE` | Hint that this is the last access |
| `NONE` | No caching |

**Synchronization protocol:**
```python
cute.copy(cp_async_atom, gmem_src, smem_dst)              # initiate
cute.arch.cp_async_commit_group()                         # commit pending copies
cute.arch.cp_async_wait_group(0)                          # wait for all groups
cute.arch.sync_threads()                                  # block-wide barrier
```

### 3. TMA Bulk Tensor Copy, `cute.nvgpu.cpasync.CopyBulkTensorTile*`

**Architecture:** SM90+ (Hopper)
**PTX:** `cp.async.bulk.tensor`

The **Tensor Memory Accelerator (TMA)** is a dedicated hardware unit on Hopper+ GPUs that can copy entire multi-dimensional tiles between GMEM and SMEM. Unlike `cp.async` (which is thread-initiated), TMA operations are issued by a single thread and the hardware handles the entire tile transfer.

| CopyOp | Path | Description |
|--------|------|-------------|
| `CopyBulkTensorTileG2SOp` | GMEM → SMEM | Bulk tensor tile load |
| `CopyBulkTensorTileG2SMulticastOp` | GMEM → SMEM (multicast) | Load + broadcast to multiple CTAs in a cluster |
| `CopyBulkTensorTileS2GOp` | SMEM → GMEM | Bulk tensor tile store |
| `CopyReduceBulkTensorTileS2GOp` | SMEM → GMEM (reduce) | Store with atomic reduction (ADD, MIN, MAX, etc.) |

```python
# GMEM → SMEM via TMA
op = cute.nvgpu.cpasync.CopyBulkTensorTileG2SOp(
    cta_group=cute.nvgpu.tcgen05.CtaGroup.ONE  # ONE or TWO (2-CTA cooperative)
)

# TMA requires a tensor map descriptor, built via make_tiled_tma_atom
tma_atom, tma_tensor = cute.nvgpu.cpasync.make_tiled_tma_atom(
    op, gmem_tensor, smem_layout, cta_tiler
)

# Execute with mbarrier synchronization
cute.copy(tma_atom, src, dst, tma_bar_ptr=mbar_ptr, mcast_mask=mask)
```

**TMA key features:**
- Single-thread issue: only one thread needs to initiate the copy
- Hardware handles address computation for multi-dimensional tiles
- Supports multicast to multiple CTAs in a cluster (SM90+)
- Uses mbarrier for synchronization instead of cp.async groups

### 4. Warp-Level Matrix Load/Store, `cute.nvgpu.warp.LdMatrix*` / `StMatrix*`

**Path:** SMEM ↔ RMEM (structured for Tensor Core consumption)
**Architecture:** SM75+ (Turing)
**PTX:** `ldmatrix`, `stmatrix`

These are **warp-level** instructions: all 32 threads in a warp cooperate to load/store a matrix tile from SMEM into registers in the exact layout that Tensor Cores expect. This avoids expensive register shuffles before MMA instructions.

#### Load Matrix (SMEM → RMEM)

| CopyOp | Matrix Shape | Element Size | Notes |
|--------|-------------|-------------|-------|
| `LdMatrix8x8x16bOp` | 8x8 | 16-bit | Basic matrix load, `.m8n8` qualifier |
| `LdMatrix8x16x8bOp` | 8x16 | 8-bit | Supports 4-bit and 6-bit unpacking |
| `LdMatrix16x8x8bOp` | 16x8 | 8-bit | Transpose required, lowers to `.m16n16` + permutation |
| `LdMatrix16x16x8bOp` | 16x16 | 8-bit | Transpose + optional unpacking (4b, 6b) |

```python
op = cute.nvgpu.warp.LdMatrix8x8x16bOp(
    transpose=False,     # whether to transpose the loaded matrix
    num_matrices=4,      # how many matrices to load (1, 2, or 4)
)
atom = cute.make_copy_atom(op, cutlass.Float16)
```

#### Store Matrix (RMEM → SMEM)

| CopyOp | Matrix Shape | Element Size | Notes |
|--------|-------------|-------------|-------|
| `StMatrix8x8x16bOp` | 8x8 | 16-bit | Basic matrix store |
| `StMatrix16x8x8bOp` | 16x8 | 8-bit | Transpose store |

```python
op = cute.nvgpu.warp.StMatrix8x8x16bOp(
    transpose=False,
    num_matrices=4,
)
atom = cute.make_copy_atom(op, cutlass.Float16)
```

**When to use:** Before/after MMA (matrix multiply-accumulate) operations. The `ldmatrix`/`stmatrix` instructions are designed to load data from SMEM into registers in exactly the layout that `mma.sync` instructions expect, avoiding costly register shuffles.

### Summary: Which CopyAtom for Which Path?

| Path | Best CopyAtom | Architecture | Use Case |
|------|--------------|-------------|----------|
| **GMEM → RMEM** | `CopyUniversalOp` | All | Element-wise ops, simple loads |
| **RMEM → GMEM** | `CopyUniversalOp` | All | Writing results back |
| **GMEM → SMEM** | `CopyG2SOp` (cp.async) | SM80+ | Staging data for block-wide reuse |
| **GMEM → SMEM** | `CopyBulkTensorTileG2SOp` (TMA) | SM90+ | Large tile loads, GEMM |
| **SMEM → GMEM** | `CopyBulkTensorTileS2GOp` (TMA) | SM90+ | Tile stores |
| **SMEM → RMEM** | `CopyUniversalOp` | All | General SMEM reads |
| **SMEM → RMEM** | `LdMatrix*Op` (ldmatrix) | SM75+ | Loading MMA operands |
| **RMEM → SMEM** | `CopyUniversalOp` | All | General SMEM writes |
| **RMEM → SMEM** | `StMatrix*Op` (stmatrix) | SM75+ | Storing MMA results |

The evolution across GPU generations is clear:
- **Turing (SM75):** Introduced `ldmatrix`/`stmatrix` for structured SMEM↔RMEM transfers
- **Ampere (SM80):** Added `cp.async` for register-free GMEM→SMEM
- **Hopper (SM90):** Added TMA for hardware-managed multi-dimensional tile transfers


## What's Next: Tensor Memory (TMEM)

Blackwell (SM100+) introduces a fifth memory tier: **Tensor Memory (TMEM)**. Unlike the four tiers covered here, TMEM is not a general-purpose scratchpad. It is a dedicated register file for the Tensor Core pipeline, only accessible via the `tcgen05` instruction family (`cp.async` SMEM→TMEM, and `ld`/`st` TMEM↔RMEM).

TMEM and the Blackwell Tensor Core programming model will be covered in a future notebook.